### Purpose

창고(이카운트 창고등록 ESA005M) 관리용 배포.

| 대상 | 내용 |
|---|---|
| DynamoDB `dswarehouses` | 키 `whcd` (창고코드) |
| Lambda `dswarehouseDynamoDB` | `/dswarehouse` GET/POST/PUT/DELETE — 목록, 엑셀 비교 반영(덮어쓰기), 삭제 |
| Lambda `dsglobaldbCreate` (갱신) | `POST /dswarehouse/create` — 새로 추가 (코드·이름 중복 409, 덮어쓰기 없음) |

- 기존 `/dschemical`, `/dscustomer` 는 건드리지 않는다
- 인증 없음 (기존 API 와 같음), 패스스루 매핑 템플릿, CORS `*`, 역할 `DSmapLambda`
- 배포 전 테스트: `pytest AWS/Lambda/dsglobaldbCreate`
- **마지막 셀(`dev` 스테이지 배포)은 API 전체를 다시 배포한다**
- 데이터 적재·갱신은 웹(창고정보 > 이카운트 엑셀 비교)에서 한다

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import json
import boto3

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
from deploy_helpers import (deploy_lambda, ensure_resource_path, get_request_template, setup_lambda_method,
                            setup_options, grant_invoke, deploy_stage_and_check)

Settings

In [ ]:
TABLE_NAME = 'dswarehouses'
TABLE_KEY = 'whcd'
ROLE_ARN = 'arn:aws:iam::199525800117:role/DSmapLambda'   # dynamodb:* on table/* 권한 있음

API_ID = 'jyipsj28s9'
STAGE = 'dev'
TEMPLATE_FROM = '/dschemical'

# Lambda → (소스, 경로별 메서드)
DEPLOY = {
    'dswarehouseDynamoDB': ('./dswarehouseDynamoDB/lambda_function.py', {'dswarehouse': ['GET', 'POST', 'PUT', 'DELETE']}),
    'dsglobaldbCreate': ('./dsglobaldbCreate/lambda_function.py', {'dswarehouse/create': ['POST']}),
}
DESCRIPTIONS = {
    'dswarehouseDynamoDB': '창고 CRUD (DynamoDB dswarehouses)',
    'dsglobaldbCreate': '글로벌 DB 안전한 추가 (약품·거래처·창고 create)',
}
ACCOUNT_ID = ROLE_ARN.split(':')[4]

1. DynamoDB 테이블 (on-demand)

In [ ]:
ddb = boto3.client('dynamodb')
if TABLE_NAME in ddb.list_tables()['TableNames']:
    print('테이블 있음:', TABLE_NAME)
else:
    ddb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{'AttributeName': TABLE_KEY, 'KeyType': 'HASH'}],
        AttributeDefinitions=[{'AttributeName': TABLE_KEY, 'AttributeType': 'S'}],
        BillingMode='PAY_PER_REQUEST',
    )
    ddb.get_waiter('table_exists').wait(TableName=TABLE_NAME)
    print('테이블 생성:', TABLE_NAME)

2. Lambda + API Gateway 경로 + 권한

In [ ]:
request_template = get_request_template(API_ID, TEMPLATE_FROM)

for function_name, (source, routes) in DEPLOY.items():
    function_arn = deploy_lambda(function_name, source, ROLE_ARN, description=DESCRIPTIONS[function_name],
                                 timeout=29, memory=256)
    for path, methods in routes.items():
        resource_id = ensure_resource_path(API_ID, path)
        for m in methods:
            setup_lambda_method(API_ID, resource_id, m, function_arn, request_template)
        setup_options(API_ID, resource_id, methods)
        grant_invoke(function_name, API_ID, ACCOUNT_ID, path)

3. `dev` 스테이지 배포 + 확인

In [ ]:
res_ = deploy_stage_and_check(API_ID, STAGE, 'dswarehouse', 'add /dswarehouse, /dswarehouse/create')
body = json.loads(res_['body'])
print('statusCode', res_['statusCode'], '/', f'{len(body)}건' if isinstance(body, list) else body)